In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 💄 AI Skincare ML Benchmark Notebook

⚡ Runs with zero external dependencies  
🧠 Covers regression, bias analysis, collaborative filtering, and deep learning recommenders  

This notebook demonstrates an end-to-end machine learning pipeline using a synthetic dermatology-inspired dataset designed for benchmarking AI systems.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model

import warnings
warnings.filterwarnings("ignore")


In [ ]:
users = pd.read_csv("/kaggle/input/ai-skincare-personalization-dataset/users.csv")
products = pd.read_csv("/kaggle/input/ai-skincare-personalization-dataset/products.csv")
interactions = pd.read_csv("/kaggle/input/ai-skincare-personalization-dataset/interactions.csv")

print(users.shape, products.shape, interactions.shape)
users.head()


In [ ]:
print("Missing Values:\n", users.isnull().sum())

users.describe()


In [ ]:
severity_cols = [c for c in users.columns if "Severity" in c]

users[severity_cols].hist(figsize=(12,8))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x="Skin_Type", data=users)
plt.xticks(rotation=30)
plt.title("Skin Type Distribution")
plt.show()

## 📊 Severity Correlation Analysis

The following heatmap shows relationships between dermatological concerns.


In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(users[severity_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Severity Correlation Matrix")
plt.show()

In [ ]:
df = users.copy()

for col in df.select_dtypes("object").columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop(columns=["Acne_Severity"])
y = df["Acne_Severity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=120, max_depth=12)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R2 Score:", r2_score(y_test, pred))


In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().tail(10).plot(kind="barh", figsize=(7,5))
plt.title("Top Features Influencing Acne Severity")
plt.show()

In [ ]:
merged_all = interactions.merge(products, on="Product_ID")

brand_bias = merged_all.groupby("Brand")["User_Rating"].mean().sort_values(ascending=False)

plt.figure(figsize=(7,4))
brand_bias.plot(kind="bar")
plt.title("Average Rating per Brand")
plt.ylabel("Avg Rating")
plt.show()


In [ ]:
ratings = interactions.pivot(
    index="User_ID",
    columns="Product_ID",
    values="User_Rating"
).fillna(0)

R = ratings.values

U, sigma, Vt = np.linalg.svd(R, full_matrices=False)
sigma = np.diag(sigma)

predicted = U @ sigma @ Vt

pred_df = pd.DataFrame(
    predicted,
    index=ratings.index,
    columns=ratings.columns
)


In [ ]:
def recommend_cf(user_id, k=5):

    user_ratings = ratings.loc[user_id]
    preds = pred_df.loc[user_id]

    unseen = user_ratings[user_ratings == 0].index
    recs = preds[unseen].sort_values(ascending=False).head(k)

    return products[products["Product_ID"].isin(recs.index)]

recommend_cf(10)


In [ ]:
df = interactions.copy()

user_enc = LabelEncoder()
item_enc = LabelEncoder()

df["user"] = user_enc.fit_transform(df["User_ID"])
df["item"] = item_enc.fit_transform(df["Product_ID"])

num_users = df["user"].nunique()
num_items = df["item"].nunique()

train, test = train_test_split(df, test_size=0.2, random_state=42)


In [ ]:
embedding_dim = 50

user_input = Input(shape=(1,))
user_vec = Flatten()(Embedding(num_users, embedding_dim)(user_input))

item_input = Input(shape=(1,))
item_vec = Flatten()(Embedding(num_items, embedding_dim)(item_input))

x = Concatenate()([user_vec, item_vec])
x = Dense(128, activation="relu")(x)
x = Dense(64, activation="relu")(x)
output = Dense(1)(x)

nn_model = Model([user_input, item_input], output)
nn_model.compile(optimizer="adam", loss="mse")

nn_model.summary()


In [ ]:
nn_model.fit(
    [train["user"], train["item"]],
    train["User_Rating"],
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)


In [ ]:
preds = nn_model.predict([test["user"], test["item"]]).flatten()
rmse_score = np.sqrt(np.mean((preds - test["User_Rating"])**2))

print("Neural Recommender RMSE:", rmse_score)

In [ ]:
def recommend_nn(user_id, k=5):

    if user_id not in user_enc.classes_:
        return "User not found in training data"

    encoded_user = user_enc.transform([user_id])[0]

    all_items = np.arange(num_items).reshape(-1,1)
    users = np.full((num_items,1), encoded_user)

    preds = nn_model.predict([users, all_items], verbose=0).flatten()

    top_items = preds.argsort()[-k:][::-1]
    original_ids = item_enc.inverse_transform(top_items)

    return products[products["Product_ID"].isin(original_ids)]


## 🏁 Conclusion

This notebook demonstrates:

- Exploratory analysis
- Regression modeling
- Bias detection
- Collaborative filtering
- Neural recommender systems

This dataset can be used as a benchmarking environment for multi-task machine learning research.

---

⭐ If you found this notebook useful, consider upvoting!
